# Spinal Muscular Atrophy Publication and Citation Records, 2016–2026 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Dataset Title:** Spinal Muscular Atrophy Publication and Citation Records, 2016–2026
- **Dataset Description:** Structured metadata and citation data for 1,438 unique publications on spinal muscular atrophy from 2016 to 2026, including source, authors, title, journal, year, DOI, and times cited, compiled from Web of Science and PubMed.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.ngp7-x46t/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section helps to understand the structure of the dataset, including its record sets and fields. All entities are referenced by their `@id` property for consistency.

In [ ]:
# List available record sets, fields and columns by '@id'
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - @id: {f['@id']} | Name: {f.get('name','N/A')} | DataType: {f.get('dataType','N/A')}")
    columns = rs.get('columns', [])
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - @id: {c['@id']} | Name: {c.get('name','N/A')} | DataType: {c.get('dataType','N/A')}")
    print("")

# For exploration: Print first record of each record set referenced by '@id'
for rs in record_sets:
    rs_id = rs['@id']
    print(f"First record for Record Set @id: {rs_id}")
    try:
        record_iter = dataset.records(record_set=rs_id)
        rec = next(record_iter)
        print(rec)
    except StopIteration:
        print("  No records found.")
    except Exception as e:
        print(f"  Error: {e}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame using each record set's `@id`. All fields and columns will be referenced by their `@id` according to the Croissant schema. DataFrames are stored in a dictionary keyed by record set `@id`.

In [ ]:
# Extract all records from each record set into a dictionary of DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
print("Extracting records for each record set by @id...")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set @id: {record_set_id} | Total records: {len(df)}")
    print(f"Columns (@id): {df.columns.tolist()}\n")

# Preview first rows from the primary record set
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"Preview for Record Set @id: {primary_rs_id}")
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. We reference the field and column `@id`s for all operations.

Below, we filter for publications with a citation count above a threshold, normalize the citation field, and group by publication year.

In [ ]:
# Identify numeric and categorical fields by their @id
primary_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[primary_rs_id]

# Example field @id's (actual values will vary, so update with correct ones from the printout above)
# Let's assume @id for 'Times cited' is 'https://api.app.sen.science/frontiers/7846712/times_cited', and for publication year: 'https://api.app.sen.science/frontiers/7846712/year'
# Update these according to your dataset's printout above!
numeric_field_id = None
group_field_id = None

# Try to infer likely numeric field (e.g. times cited)
for col in df.columns:
    if 'cited' in col.lower():
        numeric_field_id = col
    if 'year' in col.lower():
        group_field_id = col

if numeric_field_id:
    threshold = 10
    # Filter records with citation count > threshold
    filtered_df = df[df[numeric_field_id].fillna(0).astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) /
        filtered_df[numeric_field_id].astype(float).std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by year (or another group field) if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No numeric citation field found in dataset columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of citation counts and the trend of average citations per publication by year, referencing fields by their `@id`.

In [ ]:
# Plot citation distribution and trend by year
if numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    # Distribution plot
    df[numeric_field_id].fillna(0).astype(float).plot.hist(ax=ax[0], bins=30, color='skyblue')
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel("Citation Count")

    # Citations over years
    if group_field_id:
        yearly = df.groupby(group_field_id)[numeric_field_id].mean()
        yearly.plot(ax=ax[1], marker='o')
        ax[1].set_title(f"Average Citations per Publication by {group_field_id}")
        ax[1].set_xlabel("Year")
        ax[1].set_ylabel("Mean Citation Count")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric citation field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset schema enables standardized, transparent access to publication metadata and bibliometric records.
- By referencing all entities with their `@id`, data extraction and manipulation is consistent and reproducible.
- The exploratory analysis demonstrates citation record filtering, normalization, and temporal trends, supporting both bibliometric and collaboration analyses.
- Further exploration could include author network analysis, journal impact comparisons, and deep dives into publication trends for Spinal Muscular Atrophy (SMA) research during the disease-modifying therapy era (2016–2026).
